> # Notebook for data preprocessing (merging all JSON files into a single file)

In [ ]:
# Import required libraries
import json
import pandas as pd
from pathlib import Path

In [ ]:
#  Define raw data path
RAW_DATA_PATH = Path("../data/raw")

#  Map all files by domain and language
FILES = {
    "Hospitality": {
        "fr":     RAW_DATA_PATH / "Hospitality" / "Data_Frensh.json",
        "en":     RAW_DATA_PATH / "Hospitality" / "Data_English.json",
        "darija": RAW_DATA_PATH / "Hospitality" / "Data_Darija.json",
    },
    "Logistics": {
        "fr":     RAW_DATA_PATH / "Logistics" / "data_Frensh.json",
        "en":     RAW_DATA_PATH / "Logistics" / "data_English.json",
        "darija": RAW_DATA_PATH / "Logistics" / "Data_Darija.json",
    },
    "Support_Client": {
        "fr":     RAW_DATA_PATH / "Support_Client" / "Data_Frensh.json",
        "en":     RAW_DATA_PATH / "Support_Client" / "Data_English.json",
        "darija": RAW_DATA_PATH / "Support_Client" / "Data_Darija.json",
    }
}

In [ ]:
#  Function to load a JSON file
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

In [ ]:
#  Extract user messages with sentiment from each conversation
def extract_messages(data, langue, domaine):
    rows = []
    
    for conversation in data:
        #  Get conversation-level metadata
        final_sentiment  = conversation.get("final_sentiment", None)
        conversation_id  = conversation.get("id", None)
        language         = conversation.get("language", langue)
        domain           = conversation.get("domain", domaine)
        
        for turn in conversation.get("conversation", []):
            #  Keep only user messages
            if turn["speaker"] != "user":
                continue
            
            text      = turn.get("text", "").strip()
            sentiment = turn.get("sentiment", final_sentiment)
            
            #  Skip empty or unlabeled messages
            if not text or not sentiment:
                continue
            
            rows.append({
                "conversation_id" : conversation_id,
                "text"            : text,
                "sentiment"       : sentiment,
                "final_sentiment" : final_sentiment,
                "language"        : language,
                "domain"          : domain
            })
    
    return rows

In [ ]:
#  Load and merge all  JSON files into one dataframe
all_rows = []

for domain, languages in FILES.items():
    for language, path in languages.items():
        try:
            data = load_json(path)
            rows = extract_messages(data, language, domain)
            all_rows.extend(rows)
            print(f" {domain} | {language} → {len(rows)} messages extracted")
        except FileNotFoundError:
            print(f" File not found : {path}")

#  Create dataframe
df = pd.DataFrame(all_rows)
print(f"\n Total messages extracted : {len(df)}")
print(df.head(10))

In [ ]:
# Verify the dataset structure and distributions
print("=" * 40)
print("SHAPE")
print("=" * 40)
print(df.shape)

print("\n" + "=" * 40)
print("COLUMNS")
print("=" * 40)
print(df.columns.tolist())

print("\n" + "=" * 40)
print("MISSING VALUES")
print("=" * 40)
print(df.isnull().sum())

print("\n" + "=" * 40)
print("SENTIMENT DISTRIBUTION")
print("=" * 40)
print(df["sentiment"].value_counts())

print("\n" + "=" * 40)
print("LANGUAGE DISTRIBUTION")
print("=" * 40)
print(df["language"].value_counts())

print("\n" + "=" * 40)
print("DOMAIN DISTRIBUTION")
print("=" * 40)
print(df["domain"].value_counts())

SHAPE
(4271, 6)

COLUMNS
['conversation_id', 'text', 'sentiment', 'final_sentiment', 'language', 'domain']

MISSING VALUES
conversation_id    0
text               0
sentiment          0
final_sentiment    0
language           0
domain             0
dtype: int64

SENTIMENT DISTRIBUTION
sentiment
neutral      1545
positive     1397
negative     1311
surprised       8
anxious         6
worried         4
Name: count, dtype: int64

LANGUAGE DISTRIBUTION
language
fr                       1443
darija_latin             1305
en                       1073
moroccan_darija_latin     450
Name: count, dtype: int64

DOMAIN DISTRIBUTION
domain
customer_support    1746
hotellerie          1283
logistique           924
logistics            318
Name: count, dtype: int64


In [ ]:
# Save the merged dataset to CSV
OUTPUT_PATH = Path("../data/processed/dataset_clean.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print(f"Dataset saved to: {OUTPUT_PATH}")
print(f"Total rows: {len(df)}")
print(f"Columns: {df.columns.tolist()}")

Dataset saved to: ..\data\processed\dataset_clean.csv
Total rows: 4271
Columns: ['conversation_id', 'text', 'sentiment', 'final_sentiment', 'language', 'domain']


In [ ]:
#  Standardize language labels

language_mapping = {
    "fr"                    : "fr",
    "en"                    : "en",
    "darija_latin"          : "darija",
    "moroccan_darija_latin" : "darija"
}

In [ ]:
# standardize the domain labels

domain_mapping = {
    "hotellerie"      : "hotellerie",
    "logistique"      : "logistique",
    "logistics"       : "logistique",
    "customer_support": "support_client"
}

In [ ]:
# apply the mappings 
df["language"] = df["language"].map(language_mapping)
df["domain"] = df["domain"].map(domain_mapping)

print("Language and domain standardized")
print("\nLANGUAGE DISTRIBUTION AFTER STANDARDIZATION")
print(df["language"].value_counts())
print("\n DOMAIN DISTRIBUTION AFTER STANDARDIZATION")
print(df["domain"].value_counts())

Language and domain standardized

LANGUAGE DISTRIBUTION AFTER STANDARDIZATION
language
darija    1755
fr        1443
en        1073
Name: count, dtype: int64

 DOMAIN DISTRIBUTION AFTER STANDARDIZATION
domain
support_client    1746
hotellerie        1283
logistique        1242
Name: count, dtype: int64


In [ ]:
# save the final cleaned dataset to CSV

OUTPUT_PATH = Path("../data/processed/dataset_clean.csv")
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print(f"Final dataset saved → {OUTPUT_PATH}")
print(f" Total rows : {len(df)}")
print(f"\n Final distributions :")
print(f"\n Language :\n{df['language'].value_counts()}")
print(f"\n Domain :\n{df['domain'].value_counts()}")
print(f"\n Sentiment :\n{df['sentiment'].value_counts()}")

Final dataset saved → ..\data\processed\dataset_clean.csv
 Total rows : 4271

 Final distributions :

 Language :
language
darija    1755
fr        1443
en        1073
Name: count, dtype: int64

 Domain :
domain
support_client    1746
hotellerie        1283
logistique        1242
Name: count, dtype: int64

 Sentiment :
sentiment
neutral      1545
positive     1397
negative     1311
surprised       8
anxious         6
worried         4
Name: count, dtype: int64
